## Обучаем ResNet на ббоксах

## Подключаю колаб

In [1]:
!pip3 install pytorch_lightning albumentations torchvision timm rapidfuzz boto3

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/xan-d-or/sneakers-hse.git
%cd ./sneakers-hse/
!pwd
!git checkout yolo

fatal: destination path 'sneakers-hse' already exists and is not an empty directory.
/content/sneakers-hse
/content/sneakers-hse
Already on 'yolo'
Your branch is up to date with 'origin/yolo'.


In [ ]:
import os
import boto3

from concurrent.futures import ThreadPoolExecutor, as_completed

s3 = boto3.client(
    service_name="s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id="",
    aws_secret_access_key="",
)

def _download_one(bucket_name, s3_key, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(bucket_name, s3_key, local_path)
    return local_path


def download_folder_from_s3_parallel(
    bucket_name: str,
    s3_prefix: str,
    local_folder: str,
    max_workers: int = 16
):
    """
    Параллельная загрузка папки из S3
    """
    paginator = s3.get_paginator("list_objects_v2")

    tasks = []

    for page in paginator.paginate(Bucket=bucket_name, Prefix=s3_prefix):
        if "Contents" not in page:
            continue

        for obj in page["Contents"]:
            s3_key = obj["Key"]

            if s3_key.endswith("/"):
                continue

            relative_path = os.path.relpath(s3_key, s3_prefix)
            local_path = os.path.join(local_folder, relative_path)

            tasks.append((s3_key, local_path))

    print(f"Total files: {len(tasks)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_download_one, bucket_name, s3_key, local_path)
            for s3_key, local_path in tasks
        ]

        for future in as_completed(futures):
            try:
                path = future.result()
                print(f"Downloaded: {path}")
            except Exception as e:
                print(f"Error: {e}")

def _upload_one(local_path, bucket_name, s3_key):
    s3.upload_file(local_path, bucket_name, s3_key)
    return s3_key


def upload_folder_to_s3_parallel(
    local_folder: str,
    bucket_name: str,
    s3_prefix: str = "",
    max_workers: int = 16
):
    """
    Параллельная загрузка папки в S3
    """
    tasks = []

    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            tasks.append((local_path, s3_key))

    print(f"Total files: {len(tasks)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_upload_one, local_path, bucket_name, s3_key)
            for local_path, s3_key in tasks
        ]

        for future in as_completed(futures):
            try:
                s3_key = future.result()
                print(f"Uploaded: {s3_key}")
            except Exception as e:
                print(f"Error: {e}")


local_folder = '/content/sneakers-hse/data/03_yolo_preprocessed'
download_folder_from_s3_parallel(
    bucket_name='sneakers-hse-images-test',
    s3_prefix='yolo_preprocessed',
    local_folder=local_folder,
    max_workers=10
)

Total files: 11269
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_100.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_11.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/.gitkeep
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_110.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_10.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_107.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_101.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_105.jpeg
Downloaded: /content/sneakers-hse/data/03_yolo_preprocessed/search-dataset-images/Kari Кеды/clothing_0_1.jpeg
Downloaded: /content

In [4]:
!ls /content/sneakers-hse/data/03_yolo_preprocessed

03_yolo_preprocessed   test_dataset	train_images.csv
search-dataset-images  test_images.csv


In [12]:
!ls /content/sneakers-hse/data/03_yolo_preprocessed/03_yolo_preprocessed

search-dataset-images


# Готовим данные

In [13]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import pandas as pd
import numpy as np

from PIL import Image

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset
from src.model.classifier import LitClassifier

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

import fsspec

from tqdm import tqdm
tqdm.pandas()


In [14]:
train_df_pre = pd.read_csv(local_folder + '/train_images.csv')
display(train_df_pre.head(), train_df_pre.shape)

test_df = pd.read_csv(local_folder + '/test_images.csv')
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,0,Vans Кеды Upland/clothing_0_264.jpeg,Vans Кеды Upland,0
1,1,Vans Кеды Upland/clothing_0_57.jpeg,Vans Кеды Upland,0
2,2,Vans Кеды Upland/orig_45.jpeg,Vans Кеды Upland,0
3,3,Vans Кеды Upland/clothing_0_0.jpeg,Vans Кеды Upland,0
4,4,Vans Кеды Upland/clothing_0_233.jpeg,Vans Кеды Upland,0


(10965, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [15]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
2195,2288,Nike Кеды Dunk Low Retro/clothing_0_103.jpeg,Nike Кеды Dunk Low Retro,0
10557,10855,PUMA Кроссовки Puma Morphic/orig_111.jpeg,PUMA Кроссовки Puma Morphic,0
7299,7477,Kari Кроссовки/clothing_0_190.jpeg,Kari Кроссовки,0
4103,4230,Reebok Кроссовки CLASSIC LEATHER/clothing_0_43...,Reebok Кроссовки CLASSIC LEATHER,0
2097,2188,Nike Кеды Dunk Low Retro/clothing_0_86.jpeg,Nike Кеды Dunk Low Retro,0


(8772, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
7461,7639,Vans Кеды Knu Skool/clothing_0_98.jpeg,Vans Кеды Knu Skool,0
2486,2591,Reebok Кроссовки CLASSIC NYLON/orig_291.jpeg,Reebok Кроссовки CLASSIC NYLON,0
4889,5027,Nike Кроссовки AIR MAX 90/clothing_0_12.jpeg,Nike Кроссовки AIR MAX 90,0
2655,2762,Under Armour Кроссовки UA CHARGED SPEED SWIFT/...,Under Armour Кроссовки UA CHARGED SPEED SWIFT,0
231,240,Vans Кеды Upland/clothing_0_62.jpeg,Vans Кеды Upland,0


(2193, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [16]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [17]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [18]:
path_to_dataset = local_folder + '/search-dataset-images'

In [19]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [20]:
num_workers = 2
batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

In [21]:
callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3
    )
]

In [22]:
from src.data.utils.eda_utils import directory_to_dataframe

df = directory_to_dataframe(path_to_dataset)
df

,path,sneaker_class
0,Reebok Кроссовки CLASSIC NYLON/orig_293.jpeg,Reebok Кроссовки CLASSIC NYLON
1,Reebok Кроссовки CLASSIC NYLON/orig_203.jpeg,Reebok Кроссовки CLASSIC NYLON
2,Reebok Кроссовки CLASSIC NYLON/clothing_0_149....,Reebok Кроссовки CLASSIC NYLON
3,Reebok Кроссовки CLASSIC NYLON/orig_59.jpeg,Reebok Кроссовки CLASSIC NYLON
4,Reebok Кроссовки CLASSIC NYLON/clothing_0_216....,Reebok Кроссовки CLASSIC NYLON
...,...,...
11260,X-Plode Кеды/clothing_0_224.jpeg,X-Plode Кеды
11261,X-Plode Кеды/clothing_0_261.jpeg,X-Plode Кеды
11262,X-Plode Кеды/clothing_0_189.jpeg,X-Plode Кеды
11263,X-Plode Кеды/clothing_0_122.jpeg,X-Plode Кеды


In [23]:
# обучаем голову

model = LitClassifier(
    model_name="resnet18",
    num_classes=df["sneaker_class"].nunique(),
    lr=1e-3,
    freeze_backbone=True
)

trainer = pl.Trainer(
    max_epochs=5,
    callbacks=callbacks,
    enable_progress_bar=True
)

trainer.fit(model, train_loader, val_loader)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 21.0 K                                                                                           
Non-trainable params: 11.2 M                                                                                       
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 95                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [24]:
pred_batches = trainer.predict(model, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       1.00      0.25      0.40         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         4
          13       0.03      0.08      0.05        12
          14       0.00      0.00      0.00         8
          15       0.00      0.00      0.00        16
          16       0.33      0.25      0.29         4
          17       0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [25]:
pred_batches = trainer.predict(model, dataloaders=[val_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in val_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.47      0.15      0.23        54
           1       0.51      0.44      0.48        54
           2       0.49      0.76      0.59        54
           3       0.39      0.57      0.46        56
           4       0.45      0.68      0.54        56
           5       0.51      0.41      0.45        54
           6       0.63      0.47      0.53        58
           7       0.57      0.60      0.58        57
           8       0.48      0.69      0.56        54
           9       0.56      0.68      0.61        56
          10       0.39      0.57      0.46        53
          11       0.56      0.51      0.53        53
          12       0.70      0.84      0.76        56
          13       0.36      0.44      0.40        55
          14       0.51      0.69      0.59        52
          15       0.45      0.50      0.48        50
          16       0.50      0.38      0.43        55
          17       0.46    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
# уменьшаем lr и дообучаем все

model.unfreeze()
model.lr = 1e-4

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/sneakers-hse/lightning_logs/version_5/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 95                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [27]:
pred_batches = trainer.predict(model, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         4
          13       0.00      0.00      0.00        12
          14       0.00      0.00      0.00         8
          15       0.00      0.00      0.00        16
          16       0.00      0.00      0.00         4
          17       0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [28]:
pred_batches = trainer.predict(model, dataloaders=[val_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in val_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.59      0.69      0.63        54
           1       0.61      0.76      0.68        54
           2       0.73      0.81      0.77        54
           3       0.73      0.71      0.72        56
           4       0.80      0.77      0.78        56
           5       0.83      0.83      0.83        54
           6       0.84      0.83      0.83        58
           7       0.82      0.98      0.90        57
           8       0.82      0.93      0.87        54
           9       0.88      0.80      0.84        56
          10       0.76      0.89      0.82        53
          11       0.88      0.81      0.84        53
          12       0.88      0.89      0.88        56
          13       0.72      0.78      0.75        55
          14       0.85      0.87      0.86        52
          15       0.83      0.86      0.84        50
          16       0.78      0.82      0.80        55
          17       0.79    

In [31]:
!ls ./lightning_logs/

version_0  version_1  version_2  version_3  version_4  version_5  version_6


In [32]:
# Наверное текущей модели соответствует версия 5 или 6
!cp -r ./lightning_logs/ /content/drive/MyDrive/sneakers-hse

## Усиляю аугментации

In [40]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.GaussNoise(p=0.9),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=90, p=0.75),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [41]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [42]:
num_workers = 2
batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

In [43]:
callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3
    )
]

In [46]:
# обучаем голову

model_augmented = LitClassifier(
    model_name="resnet18",
    num_classes=df["sneaker_class"].nunique(),
    lr=1e-3,
    freeze_backbone=True
)

trainer = pl.Trainer(
    max_epochs=5,
    callbacks=callbacks,
    enable_progress_bar=True
)

trainer.fit(model_augmented, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 21.0 K                                                                                           
Non-trainable params: 11.2 M                                                                                       
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 95                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


In [47]:
pred_batches = trainer.predict(model_augmented, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         4
          13       0.20      0.08      0.12        12
          14       0.00      0.00      0.00         8
          15       1.00      0.06      0.12        16
          16       0.00      0.00      0.00         4
          17       0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [49]:
pred_batches = trainer.predict(model_augmented, dataloaders=[val_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in val_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        54
           1       0.00      0.00      0.00        54
           2       0.19      0.17      0.18        54
           3       0.33      0.02      0.03        56
           4       0.14      0.41      0.21        56
           5       0.14      0.06      0.08        54
           6       0.14      0.28      0.18        58
           7       0.30      0.28      0.29        57
           8       0.18      0.65      0.29        54
           9       0.60      0.11      0.18        56
          10       0.00      0.00      0.00        53
          11       0.14      0.49      0.21        53
          12       0.33      0.05      0.09        56
          13       0.00      0.00      0.00        55
          14       0.33      0.08      0.12        52
          15       0.25      0.06      0.10        50
          16       0.00      0.00      0.00        55
          17       0.67    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [50]:
# уменьшаем lr и дообучаем все

model_augmented.unfreeze()
model_augmented.lr = 1e-4

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks
)

trainer.fit(model_augmented, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/sneakers-hse/lightning_logs/version_8/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 95                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


In [51]:
pred_batches = trainer.predict(model_augmented, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       0.25      0.25      0.25         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         4
          13       0.20      0.17      0.18        12
          14       0.00      0.00      0.00         8
          15       0.00      0.00      0.00        16
          16       0.00      0.00      0.00         4
          17       0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [52]:
pred_batches = trainer.predict(model_augmented, dataloaders=[val_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in val_dataset]
print(classification_report(y_true, y_pred))


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.21      0.11      0.14        54
           1       0.39      0.28      0.33        54
           2       0.52      0.80      0.63        54
           3       0.43      0.55      0.48        56
           4       0.55      0.77      0.64        56
           5       0.45      0.48      0.46        54
           6       0.32      0.62      0.42        58
           7       0.61      0.79      0.69        57
           8       0.52      0.83      0.64        54
           9       0.62      0.70      0.66        56
          10       0.33      0.21      0.26        53
          11       0.51      0.72      0.59        53
          12       0.58      0.82      0.68        56
          13       0.24      0.15      0.18        55
          14       0.60      0.69      0.64        52
          15       0.43      0.42      0.42        50
          16       0.45      0.24      0.31        55
          17       0.76    

In [53]:
!ls ./lightning_logs/

version_0  version_2  version_4  version_6  version_8
version_1  version_3  version_5  version_7  version_9


In [54]:
# Наверное текущей модели соответствует версия 8 и 9
!cp -r ./lightning_logs/ /content/drive/MyDrive/sneakers-hse